之前介绍的 Q-learning、DQN 及 DQN 改进算法都是基于价值的方法，其中 Q-learning 是处理有限状态的算法，而 DQN 可以用来解决连续状态的问题。在强化学习中，除了基于值函数的方法，还有一支非常经典的方法，那就是基于策略的方法，对比两者，基于值函数的方法主要是学习值函数，然后根据值函数导出一个策略，学习过程中并不存在一个显式的策略；而基于策略的方法则是直接显式地学习一个目标策略。策略梯度是基于策略的方法的基础，本章从策略梯度算法说起。

基于策略的方法首先需要将策略参数化。假设目标策略是一个随机性策略，并且处处可微，其中 $\theta$ 是对应的参数;我们可以用一个线性模型或者神经网络模型来为这样一个策略函数建模，输入某个状态，然后输出一个动作的概率分布。我们的目标是要寻找一个最优策略并最大化这个策略在环境中的期望回报。我们将策略学习的目标函数定义为:
$$ J(\theta) = \mathbb{E}_{s_0} \left[ V^{\pi_0}(s_0) \right]$$

其中，表示 $s_0$ 初始状态。现在有了目标函数，我们将目标函数对策略 $\theta$ 求导，得到导数后，就可以用梯度上升方法来最大化这个目标函数，从而得到最优策略。

REINFORCE算法

In [1]:
import gym
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import rl_utils

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


首先定义策略网络PolicyNet，其输入是某个状态，输出则是该状态下的动作概率分布，这里采用在离散动作空间上的softmax()函数来实现一个可学习的多项分布

In [2]:
class PolicyNet(torch.nn.Module):
    """
        策略网络
    """
    def __init__(self, state_dim, hidden_dim, action_dim):
        """
            初始化
        """
        super(PolicyNet, self).__init__()

        self.fc1 = torch.nn.Linear(state_dim, hidden_dim)
        self.fc2 = torch.nn.Linear(hidden_dim, action_dim)

    def forward(self, x):
        """
            前向传播
            x：输入维度特征
        """
        # 第一层激活函数
        x = F.relu(self.fc1(x))
        # 输出层使用softmax函数
        return F.softmax(self.fc2(x), dim=1)

REINFORCE 算法，在函数take_action()函数中，我们通过动作概率分布对离散的动作进行采样

In [5]:
class REINFORCE:
    """
        REINFORCE算法
    """
    def __init__(self, state_dim, hidden_dim, action_dim, learning_rate, gamma, device):
        """
            初始化
            state_dim：输入层维度（状态空间的维度）
            hidden_dim：隐藏层维度
            action_dim：输出层维度
            learning_rate：学习率
            gamma：折扣因子
            device：计算运行的设备
        """
        # 定义策略网络
        self.policy_net = PolicyNet(state_dim, hidden_dim,
                                    action_dim).to(device)
        # 使用Adam优化器
        self.optimizer = torch.optim.Adam(self.policy_net.parameters(),
                                          lr=learning_rate)  
        # 折扣因子
        self.gamma = gamma  
        # 挂载设备
        self.device = device

    def take_action(self, state):
        """
            根据动作概率分布随机采样
            state：当前状态
        """
        # 将当前的环境状态 state（通常是 NumPy 数组）转换成 PyTorch 的 Tensor 格式，增加一个 batch 维度（通过 [state]），并移动到计算设备上
        state = torch.tensor([state], dtype=torch.float).to(self.device)

        # 得到当前策略网络的输出值
        probs = self.policy_net(state)

        # 利用网络输出的概率 probs，构建一个类别分布
        action_dist = torch.distributions.Categorical(probs)

        # 根据这个类别分布进行随机采样，得出一个具体的动作。概率越大的动作，被采样到的机会就越高。
        action = action_dist.sample()

        # 将 PyTorch Tensor 格式的动作提取为普通的 Python 标量（数值）并返回，以便环境执行该动作
        return action.item()
    
    def update(self, transition_dict):
        """
            策略更新
            transition_dict：从经验回放池取出的一个批次（Batch）的数据字典
        """
        # 从 transition_dict 字典中提取出整个回合的奖励列表、状态列表和动作列表
        reward_list = transition_dict['rewards']
        state_list = transition_dict['states']
        action_list = transition_dict['actions']

        # 初始回报 G 为0
        G = 0
        # 清空累积梯度
        self.optimizer.zero_grad()

        # 从最后一步算起，这里使用 MC 的方法来求 Q 值
        for i in reversed(range(len(reward_list))):  
            # 取出当前步骤 i 的奖励 reward，将对应的状态和动作转化为 PyTorch Tensor。其中动作张量使用了 .view(-1, 1) 将其形状调整为列向量，方便后续的 gather 操作
            reward = reward_list[i]
            state = torch.tensor([state_list[i]], dtype=torch.float).to(self.device)
            action = torch.tensor([action_list[i]]).view(-1, 1).to(self.device)

            # 计算对数概率
            log_prob = torch.log(self.policy_net(state).gather(1, action))

            # 计算当前时间步的回报 G
            G = self.gamma * G + reward
            # 每一步的损失函数
            loss = -log_prob * G  
            # 反向传播计算梯度
            loss.backward()  

开始实验了，看看 REINFORCE 算法在车杆环境上表现如何吧！

In [ ]:
learning_rate = 1e-3
num_episodes = 1000
hidden_dim = 128
gamma = 0.98
device = torch.device("cuda") if torch.cuda.is_available() else torch.device(
    "cpu")

env_name = "CartPole-v1"
env = gym.make(env_name)
env.seed(0)
torch.manual_seed(0)
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n
agent = REINFORCE(state_dim, hidden_dim, action_dim, learning_rate, gamma,
                  device)

return_list = []
for i in range(10):
    with tqdm(total=int(num_episodes / 10), desc='Iteration %d' % i) as pbar:
        for i_episode in range(int(num_episodes / 10)):
            episode_return = 0
            transition_dict = {
                'states': [],
                'actions': [],
                'next_states': [],
                'rewards': [],
                'dones': []
            }
            state = env.reset()
            done = False
            while not done:
                action = agent.take_action(state)
                next_state, reward, done, _ = env.step(action)
                transition_dict['states'].append(state)
                transition_dict['actions'].append(action)
                transition_dict['next_states'].append(next_state)
                transition_dict['rewards'].append(reward)
                transition_dict['dones'].append(done)
                state = next_state
                episode_return += reward
            return_list.append(episode_return)
            agent.update(transition_dict)
            if (i_episode + 1) % 10 == 0:
                pbar.set_postfix({
                    'episode':
                    '%d' % (num_episodes / 10 * i + i_episode + 1),
                    'return':
                    '%.3f' % np.mean(return_list[-10:])
                })
            pbar.update(1)